# RAG-Embeddings: Distributed Semantic Retrieval with TaskVine

This notebook extends the **RAG-Lite (BM25)** example into a full **semantic retrieval pipeline**.

Here, we use **TaskVine** to parallelize both text chunking *and* embedding generation across workers.

Each worker:
- Cleans and chunks raw text from classic books.
- Computes vector embeddings for each chunk using a lightweight model from `sentence-transformers`.

The manager:
- Collects all embeddings into a unified corpus.
- Builds a **semantic retriever** using cosine similarity.
- Demonstrates how embeddings enable deeper, context-aware retrieval — beyond keyword matching.

In [ ]:
# STEP 1: Imports & basic config

import json
from pathlib import Path

import ndcctools.taskvine as vine

print("Python + TaskVine imports OK")

# Data directory with local Gutenberg files
DATA_DIR = Path("data")

# Output directory for chunk + embedding JSONs
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Using data directory:  {DATA_DIR.resolve()}")
print(f"Using output directory: {OUTPUT_DIR.resolve()}")

# Discover input .txt files
book_paths = sorted(DATA_DIR.glob("*.txt"))

if not book_paths:
    raise RuntimeError(f"No .txt files found in {DATA_DIR}")

print("\nFound books:")
for p in book_paths:
    print(f" - {p.name} ({p.stat().st_size} bytes)")

def book_id_from_path(path: Path) -> str:
    return path.stem

book_ids = [book_id_from_path(p) for p in book_paths]
print("\nBook IDs:", book_ids)

Python + TaskVine imports OK
Using data directory:  /users/mislam5/floability-project/floability-base-dir/floability_instance_20251106_203206_885711/workflow/data
Using output directory: /users/mislam5/floability-project/floability-base-dir/floability_instance_20251106_203206_885711/workflow/output

Found books:
 - alice.txt (151191 bytes)
 - frankenstein.txt (421633 bytes)
 - pg64317.txt (306594 bytes)
 - shakespeare_complete.txt (5638525 bytes)

Book IDs: ['alice', 'frankenstein', 'pg64317', 'shakespeare_complete']


In [2]:
# STEP 2: Worker function – Stage 1: clean & chunk -> JSON file

def stage1_clean_and_chunk_book(input_filename: str, output_filename: str, book_id: str):
    """
    Stage 1 (per book): read a Gutenberg .txt file, clean it, chunk it,
    and write a list[dict] of chunks to a JSON file on the worker.

    Args:
        input_filename: staged .txt file on the worker (e.g., "book.txt")
        output_filename: JSON file to write (e.g., "chunks.json")
        book_id: identifier for the book (e.g., "1960")

    Returns:
        int: number of chunks written (for logging)
    """
    # Imports INSIDE so the function is self-contained on the worker
    import json
    import re
    from langchain_text_splitters import RecursiveCharacterTextSplitter


    # --- 1. Read full text ---
    with open(input_filename, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    # --- 2. Strip Gutenberg boilerplate (best-effort) ---

    start_markers = [
        "*** START OF THIS PROJECT GUTENBERG",
        "*** START OF THE PROJECT GUTENBERG",
        "***START OF THE PROJECT GUTENBERG",
        "*END*THE SMALL PRINT",
    ]
    for marker in start_markers:
        idx = text.find(marker)
        if idx != -1:
            text = text[idx + len(marker):]
            break

    end_markers = [
        "*** END OF THIS PROJECT GUTENBERG",
        "*** END OF THE PROJECT GUTENBERG",
        "***END OF THE PROJECT GUTENBERG",
    ]
    for marker in end_markers:
        idx = text.find(marker)
        if idx != -1:
            text = text[:idx]
            break

    # --- 3. Basic normalization ---
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text)
    text = re.sub(r" +", " ", text)
    text = text.strip()

    # --- 4. Chunking ---
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separators=["\n\n", "\n", ". ", " ", ""],
        length_function=len,
    )

    chunks = splitter.split_text(text)
    total_chunks = len(chunks)

    # --- 5. Build records ---
    records = []
    for i, chunk in enumerate(chunks):
        words = re.findall(r"\b\w+\b", chunk)
        n_words = len(words)

        if total_chunks > 1:
            relative_position = i / (total_chunks - 1)
        else:
            relative_position = 0.0

        preview = chunk[:160].replace("\n", " ")

        records.append({
            "book_id": book_id,
            "chunk_id": i,
            "total_chunks": total_chunks,
            "relative_position": relative_position,
            "text": chunk,
            "chunk_length": len(chunk),
            "n_chars": len(chunk),
            "n_words": n_words,
            "preview": preview,
        })

    # --- 6. Write JSON on worker ---
    with open(output_filename, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

    return len(records)

print("✓ Stage 1 worker stage1_clean_and_chunk_book() defined")


✓ Stage 1 worker stage1_clean_and_chunk_book() defined


In [3]:
# STEP 3: Worker function – Stage 2: chunks -> embeddings JSON

def stage2_embed_chunks(input_filename: str, output_filename: str):
    """
    Stage 2 (per book): read chunk JSON from Stage 1, compute embeddings
    for each chunk using a small, fast local model, and write out a new
    JSON with embeddings included.

    Args:
        input_filename: JSON file produced by Stage 1 (e.g., "chunks.json")
        output_filename: JSON file to write (e.g., "embeddings.json")

    Returns:
        int: number of chunks embedded
    """
    import json
    from sentence_transformers import SentenceTransformer

    # Load chunks
    with open(input_filename, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    texts = [c["text"] for c in chunks]

    if not texts:
        # Nothing to do
        with open(output_filename, "w", encoding="utf-8") as f:
            json.dump([], f, ensure_ascii=False, indent=2)
        return 0

    # Load a balanced, fast embedding model (CPU friendly)
    model = SentenceTransformer("all-MiniLM-L6-v2")

    # Compute embeddings (list of numpy arrays)
    embeddings = model.encode(texts, convert_to_numpy=True)

    # Attach embeddings to records (convert to plain lists for JSON)
    out_records = []
    for rec, emb in zip(chunks, embeddings):
        rec_with_emb = dict(rec)
        rec_with_emb["embedding"] = emb.tolist()
        out_records.append(rec_with_emb)

    # Write out
    with open(output_filename, "w", encoding="utf-8") as f:
        json.dump(out_records, f, ensure_ascii=False, indent=2)

    return len(out_records)

print("✓ Stage 2 worker stage2_embed_chunks() defined")


✓ Stage 2 worker stage2_embed_chunks() defined


In [4]:
import os

manager_name = api_key = os.environ.get("VINE_MANAGER_NAME")
print(f"Manager name: {manager_name}")

ports_str = os.environ.get("VINE_MANAGER_PORTS", "9123, 9150")
ports = [int(p.strip()) for p in ports_str.split(",")]

if len(ports) == 1:
    ports = ports[0]
else:
    ports = [int(p) for p in ports]

print(f"Manager Ports: {ports}")

Manager name: floability-74ac4815-9b6f-49fc-92c4-9e1c25b75a1b
Manager Ports: [9123, 9150]


In [5]:
import ndcctools.taskvine as vine
m = vine.Manager(ports, name=manager_name)

In [6]:
print(f"TaskVine Manager listening on port {m.port}")

TaskVine Manager listening on port 9125


In [7]:
task_meta = {}
total_tasks = 0

# For reference, track final embedding file paths
embedding_paths_by_book = {}

for path in book_paths:
    book_id = book_id_from_path(path)
    file_size = path.stat().st_size

    # Declare the input .txt file
    input_file = m.declare_file(str(path), cache="worker")

    # Intermediate temp file for chunks.json
    chunks_file = m.declare_temp()

    # Final embeddings file (local path)
    emb_output_path = OUTPUT_DIR / f"embeddings_{book_id}.json"
    embedding_paths_by_book[book_id] = emb_output_path

    emb_file = m.declare_file(str(emb_output_path))

    # --- Stage 1: raw text -> chunks.json ---
    t1 = vine.PythonTask(
        stage1_clean_and_chunk_book,
        "book.txt",      # input filename on worker
        "chunks.json",   # output filename on worker
        book_id,
    )
    t1.add_input(input_file, "book.txt")
    t1.add_output(chunks_file, "chunks.json")
    t1.set_cores(1)

    # --- Stage 2: chunks.json -> embeddings.json ---
    t2 = vine.PythonTask(
        stage2_embed_chunks,
        "chunks.json",       # input filename on worker
        "embeddings.json",   # output filename on worker
    )
    t2.add_input(chunks_file, "chunks.json")        # dependency on Stage 1
    t2.add_output(emb_file, "embeddings.json")      # final file
    t2.set_cores(1)

    id1 = m.submit(t1)
    id2 = m.submit(t2)
    total_tasks += 2

    task_meta[id1] = {"book_id": book_id, "stage": 1}
    task_meta[id2] = {"book_id": book_id, "stage": 2}

    print(f"Submitted Stage 1 (task {id1}) and Stage 2 (task {id2}) for book {book_id} ({path.name})")

print(f"\nTotal tasks submitted: {total_tasks}")

Submitted Stage 1 (task 1) and Stage 2 (task 2) for book alice (alice.txt)
Submitted Stage 1 (task 3) and Stage 2 (task 4) for book frankenstein (frankenstein.txt)
Submitted Stage 1 (task 5) and Stage 2 (task 6) for book pg64317 (pg64317.txt)
Submitted Stage 1 (task 7) and Stage 2 (task 8) for book shakespeare_complete (shakespeare_complete.txt)

Total tasks submitted: 8


In [8]:
# STEP 5: Wait for all tasks to complete

completed = 0

while not m.empty():
    t = m.wait(10)  # wait up to 10 seconds
    if not t:
        continue

    completed += 1
    meta = task_meta[t.id]
    stage = meta["stage"]
    book_id = meta["book_id"]

    if t.successful():
        if stage == 1:
            print(f"[{completed}/{total_tasks}] ✓ Stage 1 for book {book_id} (task {t.id}) "
                  f"-> {t.output} chunks")
        else:
            print(f"[{completed}/{total_tasks}] ✓ Stage 2 for book {book_id} (task {t.id}) "
                  f"-> {t.output} embeddings")
    else:
        print(f"[{completed}/{total_tasks}] Stage {stage} ✗ Task {t.id} for book {book_id} FAILED: {t.result}")

print("\nAll tasks finished.")

[1/8] ✓ Stage 1 for book alice (task 1) -> 189 chunks
[2/8] ✓ Stage 1 for book pg64317 (task 5) -> 359 chunks
[3/8] ✓ Stage 1 for book frankenstein (task 3) -> 616 chunks
[4/8] ✓ Stage 1 for book shakespeare_complete (task 7) -> 7198 chunks
[5/8] Stage 2 ✗ Task 4 for book frankenstein FAILED: output missing
[6/8] Stage 2 ✗ Task 2 for book alice FAILED: output missing
[7/8] Stage 2 ✗ Task 8 for book shakespeare_complete FAILED: output missing
[8/8] Stage 2 ✗ Task 6 for book pg64317 FAILED: output missing

All tasks finished.
